# MoveBench: Global Trajectory Map + Coverage Ideas

This notebook:

1. Loads **all studies** under `MOVEBENCH_DATA_ROOT/studies_10ind/` (i.e., the full benchmark dataset root).
2. Builds per-individual trajectory polylines.
3. Visualizes them globally on a world map (interactive via `pydeck`).
4. Brainstorms additional ways to visualize dataset **coverage** (space/time/environment).

If you haven't installed visualization deps yet, the easiest is:

```bash
pip install -e ".[apps]"
```

Notes:
- The map uses a free CARTO basemap style (no Mapbox token needed).
- For very large benchmarks, consider sampling/downsampling more aggressively.


In [ ]:
from __future__ import annotations

import os
import sys
from pathlib import Path

# --- Repo + data root wiring ---
# Run from repo root. If you're elsewhere, uncomment and set this.
# os.chdir("/path/to/movebench")

repo_root = Path.cwd().resolve()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

# If MOVEBENCH_DATA_ROOT isn't configured, default to the repo-local data path
# so Study.list_available() works out-of-the-box for this checkout.
_default_data_root = (repo_root / "data" / "movebench").resolve()
if "MOVEBENCH_DATA_ROOT" not in os.environ and _default_data_root.exists():
    os.environ["MOVEBENCH_DATA_ROOT"] = str(_default_data_root)

print("repo_root:", repo_root)
print("MOVEBENCH_DATA_ROOT:", os.environ.get("MOVEBENCH_DATA_ROOT"))


In [ ]:
import colorsys
import hashlib

import numpy as np
import pandas as pd

from movebench import Study
from movebench.trajectories import INDIVIDUAL_COL, LAT_COL, LON_COL, OUTLIER_COL, SET_COL, TIME_COL

try:
    import pydeck as pdk
except ModuleNotFoundError as e:
    raise ModuleNotFoundError(
        "pydeck is required for the interactive map. Install with: pip install -e '.[apps]' (from repo root)."
    ) from e


In [ ]:
# --- Discover studies (full benchmark = everything under studies root) ---
study_ids = Study.list_available()
print("n_studies:", len(study_ids))
study_ids[:20]


In [ ]:
# --- Load trajectories across studies ---
# Keep it lightweight: only load columns we need for mapping.
USECOLS = [INDIVIDUAL_COL, LAT_COL, LON_COL, TIME_COL, SET_COL, OUTLIER_COL]


def _load_one_study(study_id: str) -> pd.DataFrame:
    p = Study(study_id).trajectories_path

    def _use(c: str) -> bool:
        return c in USECOLS

    df = pd.read_csv(p, usecols=_use)
    df[TIME_COL] = pd.to_datetime(df[TIME_COL], errors="coerce")
    df = df.dropna(subset=[TIME_COL, LAT_COL, LON_COL])

    if OUTLIER_COL in df.columns:
        # Treat any truthy value as an outlier marker.
        df = df.loc[~df[OUTLIER_COL].fillna(False).astype(bool)].copy()

    df["study_id"] = str(study_id)
    return df


dfs = []
for sid in study_ids:
    dfs.append(_load_one_study(sid))

traj_df = pd.concat(dfs, ignore_index=True)

print("rows:", len(traj_df))
print("studies:", traj_df["study_id"].nunique())
print("individuals:", traj_df[INDIVIDUAL_COL].nunique())
print("sets:", sorted(traj_df[SET_COL].dropna().unique().tolist()))
traj_df.head()


In [ ]:
# --- Build polylines (one per (study, individual, set)) ---
MAX_POINTS_PER_PATH = 2_000
SPLIT_AT_ANTIMERIDIAN = True


def _uniform_downsample(points: list[list[float]], max_points: int) -> list[list[float]]:
    n = len(points)
    if n <= max_points:
        return points
    idx = np.linspace(0, n - 1, num=max_points, dtype=np.int64)
    return [points[int(i)] for i in idx]


def _split_antimeridian(points: list[list[float]]) -> list[list[list[float]]]:
    """Split a path when it jumps across the +/-180° seam."""
    if len(points) <= 1:
        return [points]
    out: list[list[list[float]]] = []
    cur: list[list[float]] = [points[0]]
    for lon, lat in points[1:]:
        prev_lon = cur[-1][0]
        if np.isfinite(prev_lon) and np.isfinite(lon) and abs(float(lon) - float(prev_lon)) > 180.0:
            if len(cur) >= 2:
                out.append(cur)
            cur = [[lon, lat]]
        else:
            cur.append([lon, lat])
    if len(cur) >= 2:
        out.append(cur)
    return out if out else [points]


def _stable_rgba(key: str, *, alpha: int = 180) -> list[int]:
    # Stable color assignment across runs.
    h = int(hashlib.md5(key.encode("utf-8")).hexdigest()[:8], 16) / float(0xFFFFFFFF)
    r, g, b = colorsys.hsv_to_rgb(h, 0.75, 0.9)
    return [int(round(r * 255)), int(round(g * 255)), int(round(b * 255)), int(alpha)]


path_rows: list[dict] = []

group_cols = ["study_id", INDIVIDUAL_COL, SET_COL]
for (study_id, indiv, split), g in traj_df.sort_values(TIME_COL).groupby(group_cols, sort=False):
    pts = g[[LON_COL, LAT_COL]].to_numpy(dtype=np.float64, copy=False)
    pts = pts[np.isfinite(pts).all(axis=1)]
    if len(pts) < 2:
        continue

    path = pts.tolist()  # [[lon, lat], ...]
    path = _uniform_downsample(path, MAX_POINTS_PER_PATH)

    segments = _split_antimeridian(path) if SPLIT_AT_ANTIMERIDIAN else [path]
    for seg in segments:
        if len(seg) < 2:
            continue
        path_rows.append(
            {
                "path": seg,
                "study_id": str(study_id),
                "individual": str(indiv),
                "split": str(split),
                # Color by study so studies are visually separable.
                "color": _stable_rgba(str(study_id), alpha=160 if str(split) == "train" else 220),
                "n": int(len(seg)),
            }
        )

print("n_paths (segments):", len(path_rows))
path_rows[:2]


In [ ]:
# --- Interactive global map ---
CARTO_POSITRON = "https://basemaps.cartocdn.com/gl/positron-gl-style/style.json"

layers = [
    pdk.Layer(
        "PathLayer",
        data=path_rows,
        get_path="path",
        get_color="color",
        get_width=2,
        width_min_pixels=1,
        rounded=True,
        pickable=True,
        auto_highlight=True,
    )
]

view_state = pdk.ViewState(latitude=0.0, longitude=0.0, zoom=1.15)

deck = pdk.Deck(
    layers=layers,
    initial_view_state=view_state,
    map_style=CARTO_POSITRON,
    tooltip={
        "html": "<b>{study_id}</b><br/>indiv: {individual}<br/>split: {split}<br/>points: {n}",
        "style": {"color": "white"},
    },
)

deck


In [ ]:
# --- Optional: point-density view (HexagonLayer) ---
# Helpful when the benchmark has many paths and the polyline view becomes visually dense.

points_df = traj_df[[LON_COL, LAT_COL]].rename(columns={LON_COL: "lon", LAT_COL: "lat"}).copy()

hex_layer = pdk.Layer(
    "HexagonLayer",
    data=points_df,
    get_position="[lon, lat]",
    radius=60_000,  # meters
    elevation_scale=30,
    elevation_range=[0, 3000],
    extruded=True,
    pickable=True,
)

pdk.Deck(
    layers=[hex_layer],
    initial_view_state=pdk.ViewState(latitude=0.0, longitude=0.0, zoom=1.15),
    map_style=CARTO_POSITRON,
)


In [ ]:
# --- Quick per-study coverage table (space/time) ---
summary = (
    traj_df.groupby("study_id")
    .agg(
        n_points=(LAT_COL, "size"),
        n_individuals=(INDIVIDUAL_COL, "nunique"),
        t_min=(TIME_COL, "min"),
        t_max=(TIME_COL, "max"),
        lat_min=(LAT_COL, "min"),
        lat_max=(LAT_COL, "max"),
        lon_min=(LON_COL, "min"),
        lon_max=(LON_COL, "max"),
    )
    .sort_values(["n_points"], ascending=False)
)

summary


## Coverage visualization brainstorm

Here are additional ways to visualize **coverage** of the benchmark beyond the global trajectory map.

### Spatial coverage
- **Grid occupancy**: count points per lat/lon grid cell (e.g., 1°, 2°, 5°) and plot a world heatmap.
- **Study footprints**: bounding boxes / convex hulls per study; overlap map.
- **Per-biome/eco-region**: intersect points with biome polygons (e.g., WWF ecoregions) and show counts.
- **Coastal vs inland**: histogram of distance-to-coast / distance-to-water for observed points.

### Temporal coverage
- **Seasonality**: month-of-year heatmaps per study/individual; compare train vs test.
- **Local time-of-day**: (if timestamps in local time or convertible via timezone inference) diurnal coverage.
- **Sampling cadence**: distribution of time gaps between fixes; missingness patterns.

### Movement-behavior coverage (from trajectories only)
- **Step length vs turn angle**: 2D density (behavioral space) per study and globally.
- **Burst structure**: distribution of burst lengths (continuous segments) and gap-driven breaks.

### Environmental coverage (from rasters / point feature stores)
- **Land cover classes**: histograms of MODIS LC (`modis_lc`) at observed points; compare per study & split.
- **Vegetation**: NDVI/EVI distributions; seasonal NDVI curves; tree cover fraction (MODIS CVF).
- **Climate space**: scatter/PCA/UMAP of `bioclim` variables for observed points; compare train vs test.
- **Human influence**: histogram/quantiles of Human Impact Index (`hii`).
- **Dynamic World**: per-class probability histograms; argmax class map of visited locations.

### Dataset balance & bias
- **Geographic bias**: points per continent/country (requires polygon overlay), latitudinal bands.
- **Study contribution**: Pareto chart of points/individuals per study.
- **Train/test shift**: compare spatial density and covariate distributions between splits.


## (Optional) Coverage from point feature stores (env distributions)

If you have extracted point feature stores (SSF/iSSF) with `configs/layers_all.yaml`, you can use them to compute land cover / vegetation / climate distributions at the **true** next-step endpoints.

This avoids re-sampling rasters yourself, and uses the same covariates the models see.


In [ ]:
# NOTE: This section requires torch + extracted point stores.
# It is written to fail gracefully if those aren't available yet.

try:
    import torch

    from movebench.features.manifest import read_manifest
    from movebench.features.points import select_point_store_dir
except ModuleNotFoundError as e:
    print("skip env-coverage section (missing dep):", e)
    torch = None


In [ ]:
if torch is not None:
    # Match benchmark defaults (configs/benchmarks/all.yaml)
    CONFIG_PATH = "configs/layers_all.yaml"
    SAMPLER = {
        "expected_type": "parametric",
        "num_candidates": 100,
        "random_seed": 0,
        "step_multiplier": 2.0,
        "step_q": 0.95,
        "gap_q": 0.999,
        "bounds_buffer_deg": 0.01,
    }

    # MODIS LC (IGBP) class names: 0..17
    MODIS_LC_IGBP = {
        0: "Water",
        1: "Evergreen Needleleaf Forest",
        2: "Evergreen Broadleaf Forest",
        3: "Deciduous Needleleaf Forest",
        4: "Deciduous Broadleaf Forest",
        5: "Mixed Forests",
        6: "Closed Shrublands",
        7: "Open Shrublands",
        8: "Woody Savannas",
        9: "Savannas",
        10: "Grasslands",
        11: "Permanent Wetlands",
        12: "Croplands",
        13: "Urban and Built-Up",
        14: "Cropland/Natural Vegetation Mosaic",
        15: "Snow and Ice",
        16: "Barren or Sparsely Vegetated",
        17: "Unclassified",
    }


    def load_true_env_for_study(study_id: str, *, split: str) -> tuple[list[str], np.ndarray]:
        """Return (band_ids_in_order, env_true) where env_true is (N, C) at the true endpoint."""
        study = Study(study_id)
        plan = study.layer_plan(CONFIG_PATH)
        store_dir, _m = select_point_store_dir(
            study_dir=study.study_dir,
            split=split,
            config_hash=str(plan.config_hash),
            **SAMPLER,
        )
        manifest = read_manifest(store_dir / "manifest.json")
        band_ids = manifest.layer_plan.band_ids_in_order()

        tensors = torch.load(store_dir / "features.pt", map_location="cpu")
        env = tensors["env"]  # (N, C, P)
        valid = tensors["valid"]  # (N, C, P)

        true_idx = int(env.shape[2] - 1)  # last index is the true endpoint
        env_true = env[:, :, true_idx]
        valid_true = valid[:, :, true_idx].to(torch.bool)

        env_true = torch.where(valid_true, env_true, torch.full_like(env_true, float("nan")))
        return band_ids, env_true.detach().cpu().numpy().astype(np.float32, copy=False)


    # Example: aggregate MODIS LC class histogram across all studies (train split)
    target_band = "modis_lc.LC_Type1"
    counts = {}
    for sid in study_ids:
        try:
            band_ids, env_true = load_true_env_for_study(sid, split="train")
        except Exception as e:
            print(f"[{sid}] skip (no store or load failed): {type(e).__name__}: {e}")
            continue

        if target_band not in band_ids:
            print(f"[{sid}] skip (missing band): {target_band}")
            continue

        i = band_ids.index(target_band)
        x = env_true[:, i]
        x = x[np.isfinite(x)]
        if x.size == 0:
            continue

        # LC_Type1 is categorical; coerce to ints for histogram.
        xi = x.round().astype(np.int32)
        for k, v in zip(*np.unique(xi, return_counts=True)):
            counts[int(k)] = counts.get(int(k), 0) + int(v)

    lc_hist = (
        pd.DataFrame({"class": list(counts.keys()), "count": list(counts.values())})
        .sort_values("count", ascending=False)
        .assign(name=lambda d: d["class"].map(MODIS_LC_IGBP).fillna("(unknown)"))
        .reset_index(drop=True)
    )

    lc_hist
